In [45]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='gee-space-hack')

In [46]:
# ======================== CONFIGURACION ========================

AOI = ee.FeatureCollection('projects/gee-space-hack/assets/SpaceHack/AOI_GreaterGuayaquil_v2')
ASSET_ROOT = 'projects/gee-space-hack/assets/SpaceHack'

POINTS_PER_CLASS = 1200
SEED = 42

In [47]:
# ======================== CARGAR DATASETS ========================

# Manglares SERVIR 2022 (FeatureCollection / shapefile)
man2022 = ee.FeatureCollection('projects/gee-space-hack/assets/SpaceHack/data/MAN_2022')

# Rasterizar MAN_2022 para crear mascara binaria (1=manglar, 0=no manglar)
mangrove_raster = (ee.Image(0)
    .paint(man2022, 1)
    .rename('manglar')
    .clip(AOI.geometry()))

# ESA WorldCover 2021 (10m)
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(AOI.geometry())

# JRC Global Surface Water
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(AOI.geometry())
water_mask = jrc_water.gt(80)

# Global Human Settlement Layer 2020
ghsl = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_S/2020').clip(AOI.geometry())
urban_mask = ghsl.select('built_surface').gt(0)



In [48]:
# ======================== MASCARAS POR CLASE ========================

# Clase 1: Manglar (desde SERVIR MAN_2022)
class1_mask = mangrove_raster

# Clase 2: Otra vegetacion (arboles, cultivos, pastos - excluyendo manglar y urbano)
other_veg = worldcover.eq(10).Or(worldcover.eq(20)).Or(worldcover.eq(30)).Or(worldcover.eq(40))
class2_mask = other_veg.And(mangrove_raster.Not()).And(urban_mask.Not())

# Clase 3: Agua (JRC permanente, excluyendo manglar)
class3_mask = water_mask.And(mangrove_raster.Not())

# Clase 4: Urbano (GHSL, excluyendo manglar y agua)
class4_mask = urban_mask.And(mangrove_raster.Not()).And(water_mask.Not())

# Clase 5: Suelo desnudo / camaroneras
bare_soil = worldcover.eq(60)
wetland_non_mangrove = worldcover.eq(90).And(mangrove_raster.Not())
class5_mask = bare_soil.Or(wetland_non_mangrove).And(urban_mask.Not()).And(water_mask.Not())


In [49]:
# ======================== MUESTREO ========================

# Imagen combinada con todas las clases
class_image = (class1_mask.multiply(1)
    .add(class2_mask.multiply(2))
    .add(class3_mask.multiply(3))
    .add(class4_mask.multiply(4))
    .add(class5_mask.multiply(5))
    .rename('class'))

# Donde hay superposicion entre mascaras, la prioridad es:
# manglar > agua > urbano > vegetacion > suelo
# Reclasificar con prioridad explicita
priority_class = (ee.Image(0)
    .where(class5_mask, 5)
    .where(class2_mask, 2)
    .where(class4_mask, 4)
    .where(class3_mask, 3)
    .where(class1_mask, 1)
    .rename('class')
    .clip(AOI.geometry()))

# Muestreo estratificado
training_points = priority_class.selfMask().stratifiedSample(
    numPoints=POINTS_PER_CLASS,
    classBand='class',
    region=AOI.geometry(),
    scale=10,
    seed=SEED,
    geometries=True
)


In [50]:
# ======================== VERIFICACION ========================

# Contar puntos por clase
for cls, name in [(1, 'Manglar'), (2, 'Otra vegetacion'), (3, 'Agua'),
                  (4, 'Urbano'), (5, 'Suelo/Camaronera')]:
    n = training_points.filter(ee.Filter.eq('class', cls)).size().getInfo()
    print(f'  {name}: {n}')

total = training_points.size().getInfo()
print(f'  TOTAL: {total}')


  Manglar: 1200
  Otra vegetacion: 1200
  Agua: 1200
  Urbano: 1200
  Suelo/Camaronera: 1200
  TOTAL: 6000


In [51]:
# ======================== EXPORTAR ========================

task = ee.batch.Export.table.toAsset(**{
    'collection': training_points,
    'description': 'training_points_v3',
    'assetId': f'{ASSET_ROOT}/training_points_v3'
})
task.start()
print(f'\nExport lanzado: {ASSET_ROOT}/training_points_v3')



Export lanzado: projects/gee-space-hack/assets/SpaceHack/training_points_v3


In [52]:
# ======================== VISUALIZACION ========================

# Compuesto S2 2022 como fondo (usar asset si ya exportaste, sino generar rapido)
s2_bg = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI.geometry())
    .filterDate('2022-03-01', '2022-12-31')
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 30))
    .median()
    .clip(AOI.geometry()))

Map = geemap.Map(center=[-2.2, -79.9], zoom=11)

Map.addLayer(s2_bg, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'S2 RGB 2022')
Map.addLayer(s2_bg, {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 4000}, 'S2 Falso Color', shown=False)

# Manglar SERVIR como referencia
Map.addLayer(man2022, {'color': '00ff8855'}, 'MAN_2022 SERVIR', shown=False)

# Mascara de clases
vis_classes = {'min': 1, 'max': 5, 'palette': ['00cc44', 'cccc00', '0066ff', 'ff3333', 'ff8800']}
Map.addLayer(priority_class.selfMask(), vis_classes, 'Mascaras de clase', shown=False)

# Puntos por clase
colors = {1: 'green', 2: 'yellow', 3: 'blue', 4: 'red', 5: 'orange'}
names = {1: 'Manglar', 2: 'Otra veg', 3: 'Agua', 4: 'Urbano', 5: 'Suelo'}

for cls in range(1, 6):
    pts = training_points.filter(ee.Filter.eq('class', cls))
    Map.addLayer(pts, {'color': colors[cls]}, f'Pts {names[cls]}')

Map.addLayerControl()
Map

Map(center=[-2.2, -79.9], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…